In [ ]:
# Imports
from pathlib import Path
import itertools
import os
import pickle

import numpy as np
import pandas as pd
from brian2 import device
from joblib import Parallel, delayed

from simulator_with_force_long_AHP_output_force_of_different_pools_separately import (
    finalize_step_response_batch,
    finalize_step_response_batch_mean_traces,
    SimulationParameters,
    run_simulation,
)


In [ ]:
# STEP-RESPONSE BATCH CONFIGURATION
batch_name_original = "$$$$_sim_batches_STEP_RESPONSE_with_repeats_V1"
add_to_same_batch_simulation_folder = True # False
RUN_SIMULATIONS = False # True
LOAD_EXISTING_RUN_OUTPUT_TABLE_WHEN_NOT_SIMULATING = True
SIM_PARALLEL_CPUS = 16
FINALIZE_STEP_RESPONSE_REPEATS = True
FINALIZATION_N_JOBS = 1
REGENERATE_STEP_RESPONSE_FIGURES_ON_FINALIZE = True
FINALIZE_STEP_RESPONSE_MEAN_TRACES = True
MEAN_TRACE_FINALIZATION_N_JOBS = 1
REGENERATE_MEAN_TRACE_FIGURES = True

# Each baseline-grid row defines one pre-step state. Step amplitudes are expanded
# within each repeat using the same requested_random_seed, so nonzero steps can
# later be compared against the matched zero-step reference.
BASELINE_PARAMETER_GRID = {
    "excitatory_input_baseline": [38_000.0],
    # # ^ 38_000 = ~ roughly the value from the baseline posterior inference
    "lf_target_sd": [2_500.0], # [0.0, 1000.0, 2500.0],
    # # ^ 2_500 = ~ roughly the value from the baseline posterior inference
    # "alpha_target_sd_final": [0.0, 1000.0, 2500.0, 5000.0], # [2_500.0], # [0.0, 1000.0, 2500.0, 5000.0], # , 500.0],
    # # ^ 2_700 = ~ roughly the value from the baseline posterior inference
    # "beta_target_sd_final": [0.0, 1000.0, 2500.0, 5000.0], # [0.0, 1000.0, 2500.0, 5000.0], # , 500.0],
    # # ^ 4_000 = ~ roughly the value from the baseline posterior inference
    "alpha_beta_envelope_variability_scale": [0.0],
    # # ^ 1 (full variability) is the default used for the SBI inference (reproduce quite well experimental observations)
}

# Optional uniform sampling for baseline parameter-set values.
# Sampled values are drawn once per parameter_set_id and then reused unchanged
# across repeats and across all paired step amplitudes.
BASELINE_PARAMETER_UNIFORM_SPECS = {
    # "lf_target_sd": {"low": 0.0, "high": 5_000.0},
    "alpha_target_sd_final": {"low": 0.0, "high": 5_000.0},
    "beta_target_sd_final": {"low": 0.0, "high": 5_000.0},
}
N_UNIFORM_SAMPLES_PER_GRID_ROW = 100 # 128
PARAMETER_SET_SAMPLING_SEED = 24680
STEP_CURRENT_AMPLITUDES_NA = [20_000] # [-10_000, 10_000, 25_000] # [-10_000, -5_000, 5_000, 10_000, 15_000] #[-10_000.0, -5_000.0, 5_000.0, 10_000.0, 15_000.0] # Always 0 added by default (baseline + reference)
INCLUDE_ZERO_STEP_REFERENCE = True

repeats_per_parameter_set = 10 # 10 # Allow to get averaged-out curves over this many repeats
batch_repeat_seed = 12345

# Step diagnostic figure style. Axis limits are optional; leave as None for autoscaling.
STEP_DIAGNOSTIC_PLOT_STYLE = {
    "figsize": [12.0, 7.0],
    "dpi": 140,
    "xlim_rel_s": None,        # e.g. [-3.0, 3.0] None is the entire duration
    "input_ylim_uA": [0.0, 100.0], # None, # [0.0, 80.0], # None,     # left y-axis, e.g. [34.0, 46.0]
    "cst_ylim": [0.0, 25.0], # None,          # right y-axis, e.g. [0.0, 30.0]
    "show_cst_lf": False,
    "show_cst_hann": False,
    "show_cst_causal": True,
    "show_matched_no_step": True,
    "show_step_effect": True,
    "show_slope_fit": True,
    "show_response_shape_fit": True,
    "show_response_shape_fit_window": False,
    "show_xcorr_window": False,
    "show_sync_predictor_windows": True,
    "show_step_current_trace": False,
    "show_step_onset_line": True,
    "show_sync_subplot": True,
    "slope_fit_overlay_mode": "cst",  # "cst" or "effect"
    "total_input_color": "#0066ff",
    "baseline_input_color": "#0066ff",
    "step_current_color": "#111111",
    "cst_color": "#ff3c00",
    "cst_lf_color": "#ff3c00",
    "cst_causal_color": "#ff3c00",
    "matched_no_step_color": "#ff9100",
    "step_effect_color": "#ff9100",
    "slope_fit_color": "#000000",
    "response_shape_fit_color": "#000000",
    "response_shape_fit_window_color": "#008c72",
    "xcorr_window_color": "#818181",
    "sync_predictor_pre_window_color": "#C350CE",
    "sync_predictor_local_window_color": "#C350CE",
    "sync_trace_color": "#852680",
    "matched_no_step_sync_color": "#777777",
    "sync_effect_color": "#111111",
    "cst_hann_color": "#ff3300",
    "step_line_color": "#555555",
    "pre_window_color": "#6babff",
    "late_post_window_color": "#ffdb3d",
    "total_input_lw": 1.0,
    "baseline_input_lw": 1.0,
    "step_current_lw": 1.0,
    "cst_lw": 1.5,
    "cst_lf_lw": 1.0,
    "cst_causal_lw": 1.0,
    "matched_no_step_lw": 1.1,
    "step_effect_lw": 1.0,
    "slope_fit_lw": 2.0,
    "response_shape_fit_lw": 1.6,
    "sync_trace_lw": 1.4,
    "matched_no_step_sync_lw": 1.1,
    "sync_effect_lw": 1.2,
    "cst_hann_lw": 1.3,
    "step_line_lw": 1.1,
    "baseline_input_ls": "-",
    "step_current_ls": "-",
    "cst_lf_ls": "-",
    "matched_no_step_ls": "-",
    "step_effect_ls": "-",
    "slope_fit_ls": "-",
    "response_shape_fit_ls": "--",
    "matched_no_step_sync_ls": "--",
    "sync_effect_ls": "-",
    "cst_hann_ls": "-",
    "step_line_ls": "--",
    "baseline_input_alpha": 0.5,
    "total_input_alpha": 0.5,
    "step_current_alpha": 1.0,
    "cst_alpha": 1.0,
    "cst_lf_alpha": 0.5,
    "cst_causal_alpha": 1.0,
    "matched_no_step_alpha": 0.5,
    "step_effect_alpha": 0.7,
    "slope_fit_alpha": 0.7,
    "response_shape_fit_alpha": 0.95,
    "sync_trace_alpha": 1.0,
    "matched_no_step_sync_alpha": 0.45,
    "sync_effect_alpha": 0.95,
    "cst_hann_alpha": 0.9,
    "pre_window_alpha": 0.12,
    "late_post_window_alpha": 0.12,
    "response_shape_fit_window_alpha": 0.08,
    "xcorr_window_alpha": 0.06,
    "sync_predictor_pre_window_alpha": 0.12,
    "sync_predictor_local_window_alpha": 0.08,
    "grid_alpha": 0.25,
    "legend_loc": "lower left",
    "legend_fontsize": 7.0,
    "annotation_fontsize": 7.0,
    "sync_annotation_fontsize": 7.0,
    "sync_annotation_loc": "upper left",
    "title": "Step-response diagnostic",
}

MEAN_TRACE_DIAGNOSTIC_PLOT_STYLE = {
    "figsize": [11.5, 6.8],
    "dpi": 140,
    "xlim_rel_s": STEP_DIAGNOSTIC_PLOT_STYLE.get("xlim_rel_s"),
    "cst_ylim": None,
    "sync_ylim": None,
    "show_individual_traces": True,
    "individual_trace_alpha": 0.16,
    "individual_trace_lw": 0.8,
    "mean_trace_alpha": 1.0,
    "mean_trace_lw": 2.2,
    "step_effect_color": STEP_DIAGNOSTIC_PLOT_STYLE.get("step_effect_color", "#ff9100"),
    "sync_trace_mode": "raw",  # "raw" or "effect"/"residual"
    "sync_trace_color": STEP_DIAGNOSTIC_PLOT_STYLE.get("sync_trace_color", "#852680"),
    "sync_effect_color": STEP_DIAGNOSTIC_PLOT_STYLE.get("sync_effect_color", "#111111"),
    "slope_fit_color": STEP_DIAGNOSTIC_PLOT_STYLE.get("slope_fit_color", "#000000"),
    "response_shape_fit_color": STEP_DIAGNOSTIC_PLOT_STYLE.get("response_shape_fit_color", "#000000"),
    "pre_window_color": STEP_DIAGNOSTIC_PLOT_STYLE.get("pre_window_color", "#6babff"),
    "late_post_window_color": STEP_DIAGNOSTIC_PLOT_STYLE.get("late_post_window_color", "#ffdb3d"),
    "pre_window_alpha": STEP_DIAGNOSTIC_PLOT_STYLE.get("pre_window_alpha", 0.12),
    "late_post_window_alpha": STEP_DIAGNOSTIC_PLOT_STYLE.get("late_post_window_alpha", 0.12),
    "show_slope_fit": True,
    "show_response_shape_fit": True,
    "legend_loc": "upper right",
    "sync_legend_loc": "upper right",
    "legend_fontsize": 7.0,
    "annotation_fontsize": 7.0,
    "sync_annotation_fontsize": 7.0,
    "show_sync_zero_line": False,
    "grid_alpha": 0.25,
    "title": "Mean-trace step-response diagnostic",
}

# Keep this workflow independent from force-target optimization and Go/No-Go bursts.
FIXED_PARAMETERS = {
    "brian_codegen_target": "cython", # "numpy"

    # DIAGNOSTIC OUTPUT CONTROL
    "output_plots": True,
    "output_plot_summary_input_cst_figure": False,
    "output_plot_input_component_figures": True,
    "output_plot_common_input_weight_figure": False,
    "output_plot_independent_input_weight_figure": False,
    "output_plot_sync_diagnostic_figure": False,
    "output_plot_firing_rate_figure": True,
    "output_plot_spike_trains_figure": True,
    "output_plot_mn_size_figure": True,
    "output_plot_mn_properties_figure": False,
    "output_plot_target_baseline_figure": False,
    "output_plot_common_vs_independent_power_figure": True,
    "output_plot_force_tracking_figure": False,
    "output_plot_force_properties_figures": False,
    "output_plot_force_optimization_figures": False,
    "save_common_input_power_csv": False,
    "save_independent_input_power_csv": False,

    # CST analysis
    "enable_cst_hann_smoothing": True,
    "cst_hann_smoothing_window_s": 0.1, # 0.4,
    "step_response_cst_smoothing_method": "causal_hann",  # "causal_exponential" or "causal_hann"
    "step_response_cst_smoothing_tau_s": 0.1, # Used only when method is "causal_exponential".
    "step_response_cst_hann_half_width_s": 0.15, # 0.080,

    # MINIMAL OUTPUT PROFILE
    "minimal_output": False,
    "minimal_output_turn_off_figure_output": True, # True,
    "minimal_output_save_spike_trains": False,
    "minimal_output_downsample_saved_traces": True,
    "minimal_output_target_fsamp": 100,

    "duration": 4.0,
    "fsamp": 2000,
    "edges_ignore_duration": 1.0,
    # "get_ready_cue_time_s": 2.0, # Not used here
    # "go_nogo_cue_delay_s": 1.0, # Not used here
    # STEP CURRENT PARAMETERS
    "enable_step_current": True,
    "step_current_time_s": 2.0,
    "step_current_duration_s": None,
    "step_current_mode": "absolute_nA",
    "step_current_amplitude_percent_baseline": None,
    "step_current_ramp_time_s": 0.0,
    "step_pre_window_rel_s": [-2.0, -0.5],
    "step_late_post_window_rel_s": [0.5, 2.0],
    "step_slope_candidate_min_s": 0.050,
    "step_slope_candidate_max_s": 0.500,
    "step_slope_candidate_step_s": 0.010,
    "step_slope_r2_min": 0.90,
    "step_slope_response_fraction_min": 0.25,
    "response_shape_fit_window_rel_s": [0.0, 0.5],
    "response_shape_tau_min_ms": 1.0,
    "response_shape_tau_max_ms": 5000.0,
    "response_shape_min_r_inf": 1e-9,
    "xcorr_lag_window_rel_s": [0.0, 1.0],
    "xcorr_max_lag_s": 0.25,
    "sync_predictor_pre_step_window_rel_s": [-0.100, 0.000], # [-0.250, 0.000],
    "sync_predictor_local_step_window_rel_s": [-0.050, 0.050], # [-0.250, 0.250],
    "enable_step_sync_index_analysis": True,
    "step_sync_analysis_window_rel_s": [-2.0, 2.0], # full window on which to compute synchrony
    "step_diagnostic_display_window_rel_s": [-2, 2], # just for display
    "step_diagnostic_plot_style": STEP_DIAGNOSTIC_PLOT_STYLE,
    "output_plot_step_response_diagnostic_figure": True,
    "enable_go_nogo_cue_diagnostics": False,
    "spike_train_show_task_cues_when_no_burst": False,
    "spike_train_show_step_current_marker": True,

    # MOTOR-UNIT POPULATION
    "nb_motoneurons": 40,
    "min_soma_diameter": 50,
    "max_soma_diameter": 100,
    "tau_size_mode": "tau_size_um",
    "tau_size_ratio": 0.2,
    "tau_size_um": 4.0, # ~ roughly the value from the baseline posterior inference

    # COMMON AND INDEPENDENT INPUTS
    "excitatory_input_baseline": 38 * 1e3, # ~ roughly the value from the baseline posterior inference
    "lf_target_sd": 2.5 * 1e3, # ~ roughly the value from the baseline posterior inference
    "lf_band_hz": [0.0, 5.0],
    "alpha_target_sd_final": 2.7 * 1e3, # ~ roughly the value from the baseline posterior inference
    "alpha_band_hz": [8.0, 13.0],
    "alpha_envelope_band_hz": [0.0, 3.0],
    "beta_target_sd_final": 4.0 * 1e3, # ~ roughly the value from the baseline posterior inference
    "beta_band_hz": [13.0, 30.0],
    "beta_envelope_band_hz": [0.0, 10.0],
    "alpha_beta_envelope_variability_scale": 0.0, # 1.0, if 0 = no envelope, just a flat x1 multiplier (so directly the filtered gaussian noise with required SD); if 1 = full envelope, with troughs and peaks multiplier (still mean of 1, but sometimes the net alpha/beta modulation is at 0)
    "common_input_weight_sd": 0.8, # ~ roughly the value from the baseline posterior inference
    "common_input_weight_min": 0.0,
    "common_input_weight_max": 10.0,
    "independent_input_power": 5.0 * 1e3, # ~ roughly the value from the baseline posterior inference # if independent_input_absolute_or_ratio == "absolute", this is in nA as sd (like the other inputs); if independent_input_absolute_or_ratio == "ratio", this is relative to common input power   
    "low_pass_filter_of_MN_independent_input": 50.0,
    "independent_input_weight_sd": 0.7, # ~ roughly the value from the baseline posterior inference
    "independent_input_weight_max": 10.0,
    "independent_input_absolute_or_ratio": "absolute", # power expressed in nA

    # Step synchrony uses step_pre_window_rel_s and step_late_post_window_rel_s.
    "enable_sync_index_analysis": False,
    "sync_win_ms": 100.0,
    "sync_step_ms": 10.0,
    "sync_coinc_lag_ms": 6.0,
    "sync_direction_mode": "legacy_forward",
    "sync_expectation_mode": "analytic",

    # FORCE OUTPUT (DISABLED)
    "enable_force_model": False,
    "variable_excitatory_input_baseline": False,
    "optimize_baseline": False,
    "save_per_mu_forces": False,
    "force_kernel_duration_factor": 6.0,
    "tetanic_force_at_min_diam": 1.0,
    "tetanic_force_at_max_diam": 10.0,
    "twitch_peak_rel_at_min_diam": 0.20,
    "twitch_peak_rel_at_max_diam": 0.80,
    "twitch_time_at_min_diam": 0.10,
    "twitch_time_at_max_diam": 0.02,
    "force_diameter_exponent": 2.0,

    # Burst/trend go-nogo simulation inputs and analyses (DISABLED)
    "enable_input_burst": False,
    "lf_burst_peak_nA": 0.0,
    "alpha_burst_peak_nA": 0.0,
    "beta_burst_peak_nA": 0.0,
    "lf_trend_slope_nA_per_s": 0.0,
    "alpha_trend_slope_nA_per_s": 0.0,
    "beta_trend_slope_nA_per_s": 0.0,


}


In [ ]:
# Build batch folders and paired step/repeat parameter tables.
def resolve_batch_name(base_name: str, add_to_same_folder: bool) -> str:
    base = Path(base_name)
    if add_to_same_folder or not base.exists():
        base.mkdir(parents=True, exist_ok=True)
        return str(base)
    idx = 1
    while True:
        candidate = Path(f"{base_name}__run_{idx:04d}")
        if not candidate.exists():
            candidate.mkdir(parents=True, exist_ok=False)
            return str(candidate)
        idx += 1


def build_grid_dataframe(parameter_grid: dict) -> pd.DataFrame:
    keys = list(parameter_grid.keys())
    rows = [dict(zip(keys, values)) for values in itertools.product(*(parameter_grid[k] for k in keys))]
    return pd.DataFrame(rows)


def normalize_uniform_spec(parameter_name: str, spec) -> tuple[float, float]:
    if isinstance(spec, dict):
        low = spec.get("low")
        high = spec.get("high")
    else:
        try:
            low, high = spec
        except Exception as exc:
            raise ValueError(
                f"Uniform spec for {parameter_name!r} must be a dict with low/high or a two-item sequence"
            ) from exc
    low = float(low)
    high = float(high)
    if not np.isfinite(low) or not np.isfinite(high) or not low < high:
        raise ValueError(f"Uniform spec for {parameter_name!r} must satisfy finite low < high; got {low}, {high}")
    return low, high


def validate_parameter_sampling_config(parameter_grid: dict, uniform_specs: dict, n_samples_per_grid_row: int) -> dict:
    overlapping = sorted(set(parameter_grid).intersection(uniform_specs))
    if overlapping:
        raise ValueError(
            "A parameter cannot be listed in both BASELINE_PARAMETER_GRID and BASELINE_PARAMETER_UNIFORM_SPECS: "
            f"{overlapping}"
        )
    n_samples_per_grid_row = int(n_samples_per_grid_row)
    if n_samples_per_grid_row < 1:
        raise ValueError("N_UNIFORM_SAMPLES_PER_GRID_ROW must be >= 1")
    return {
        name: {"low": low, "high": high}
        for name, spec in uniform_specs.items()
        for low, high in [normalize_uniform_spec(name, spec)]
    }


def build_mixed_baseline_parameter_sets(
    parameter_grid: dict,
    uniform_specs: dict,
    *,
    n_uniform_samples_per_grid_row: int = 1,
    sampling_seed: int = 0,
) -> tuple[pd.DataFrame, dict]:
    normalized_uniform_specs = validate_parameter_sampling_config(
        parameter_grid,
        uniform_specs,
        n_uniform_samples_per_grid_row,
    )
    grid_df = build_grid_dataframe(parameter_grid)
    rng = np.random.default_rng(int(sampling_seed))
    rows = []
    for grid_row_id, (_, grid_row) in enumerate(grid_df.iterrows()):
        base_item = grid_row.to_dict()
        for uniform_sample_index in range(int(n_uniform_samples_per_grid_row)):
            item = dict(base_item)
            for parameter_name, spec in normalized_uniform_specs.items():
                item[parameter_name] = float(rng.uniform(spec["low"], spec["high"]))
            item["grid_row_id"] = int(grid_row_id)
            item["uniform_sample_index"] = int(uniform_sample_index)
            item["parameter_set_sampling_seed"] = int(sampling_seed)
            rows.append(item)
    return pd.DataFrame(rows), normalized_uniform_specs


def resolve_step_amplitudes(step_amplitudes, include_zero_reference=True):
    resolved = []
    if include_zero_reference:
        resolved.append(0.0)
    resolved.extend(float(value) for value in step_amplitudes)
    deduped = []
    for value in resolved:
        if not any(np.isclose(value, existing, atol=1e-9, rtol=0.0) for existing in deduped):
            deduped.append(float(value))
    return deduped


def format_step_folder_name(step_amplitude_nA: float) -> str:
    amplitude = float(step_amplitude_nA)
    abs_amplitude = abs(amplitude)
    prefix = "m" if amplitude < 0 else "p"
    if np.isclose(abs_amplitude, round(abs_amplitude), atol=1e-9, rtol=0.0):
        label = f"{int(round(abs_amplitude)):05d}"
    else:
        label = f"{abs_amplitude:.6f}".rstrip("0").rstrip(".").replace(".", "p")
    return f"step_{prefix}{label}_nA"


batch_name = resolve_batch_name(
    batch_name_original,
    add_to_same_batch_simulation_folder or (not RUN_SIMULATIONS and LOAD_EXISTING_RUN_OUTPUT_TABLE_WHEN_NOT_SIMULATING),
)
step_current_amplitudes_nA = resolve_step_amplitudes(
    STEP_CURRENT_AMPLITUDES_NA,
    include_zero_reference=INCLUDE_ZERO_STEP_REFERENCE,
)
baseline_parameter_sets_df, normalized_baseline_parameter_uniform_specs = build_mixed_baseline_parameter_sets(
    BASELINE_PARAMETER_GRID,
    BASELINE_PARAMETER_UNIFORM_SPECS,
    n_uniform_samples_per_grid_row=N_UNIFORM_SAMPLES_PER_GRID_ROW,
    sampling_seed=PARAMETER_SET_SAMPLING_SEED,
)
baseline_parameter_sets_df.insert(0, "parameter_set_id", np.arange(len(baseline_parameter_sets_df), dtype=int))
baseline_parameter_sets_df["parameter_set_folder_name"] = baseline_parameter_sets_df["parameter_set_id"].map(lambda i: f"parameter_set_{int(i):04d}")
parameter_sets_df = baseline_parameter_sets_df.copy()

rng = np.random.default_rng(batch_repeat_seed)
paired_repeat_rows = []
prior_rows = []
for _, row in baseline_parameter_sets_df.iterrows():
    for repeat_index in range(int(repeats_per_parameter_set)):
        repeat_seed = int(rng.integers(0, 2**32 - 1, dtype=np.uint32))
        repeat_folder_name = f"repeat_{int(repeat_index):04d}"
        repeat_item = row.to_dict()
        repeat_item.update({
            "repeat_index": int(repeat_index),
            "repeat_folder_name": repeat_folder_name,
            "paired_repeat_seed": repeat_seed,
            "requested_random_seed": repeat_seed,
        })
        paired_repeat_rows.append(repeat_item)
        reference_step_folder_name = format_step_folder_name(0.0)
        for step_index, step_amplitude_nA in enumerate(step_current_amplitudes_nA):
            item = repeat_item.copy()
            step_folder_name = format_step_folder_name(step_amplitude_nA)
            item.update({
                "step_index": int(step_index),
                "step_current_amplitude_nA": float(step_amplitude_nA),
                "is_zero_step_reference": bool(np.isclose(step_amplitude_nA, 0.0, atol=1e-9, rtol=0.0)),
                "paired_reference_folder_name": reference_step_folder_name,
                "step_folder_name": step_folder_name,
            })
            prior_rows.append(item)
paired_repeat_seed_df = pd.DataFrame(paired_repeat_rows)
prior_samples_df = pd.DataFrame(prior_rows)

print(f"Batch folder: {batch_name}")
print(f"Baseline parameter sets: {len(baseline_parameter_sets_df)}")
print(f"Grid rows: {baseline_parameter_sets_df['grid_row_id'].nunique() if 'grid_row_id' in baseline_parameter_sets_df else 'NA'}")
print(f"Uniform samples per grid row: {N_UNIFORM_SAMPLES_PER_GRID_ROW}")
print(f"Uniform-sampled parameters: {list(normalized_baseline_parameter_uniform_specs)}")
print(f"Paired repeat seeds: {len(paired_repeat_seed_df)}")
print(f"Step amplitudes: {step_current_amplitudes_nA}")
print(f"Total simulations: {len(prior_samples_df)}")
baseline_parameter_sets_df.head()


In [ ]:
# Build SimulationParameters objects and save the batch prior table.
PARAMETER_SET_TABLE_METADATA_COLUMNS = {
    "grid_row_id",
    "uniform_sample_index",
    "parameter_set_sampling_seed",
}


def row_to_simulation_parameters(row: pd.Series) -> SimulationParameters:
    row_dict = dict(row)
    row_dict["output_folder_name"] = str(
        Path(batch_name)
        / str(row_dict["parameter_set_folder_name"])
        / str(row_dict["repeat_folder_name"])
        / str(row_dict["step_folder_name"])
    )
    for metadata_key in PARAMETER_SET_TABLE_METADATA_COLUMNS:
        row_dict.pop(metadata_key, None)
    init_kwargs = dict(FIXED_PARAMETERS)
    init_kwargs.update(row_dict)
    return SimulationParameters(**init_kwargs)


params_prior_list = [row_to_simulation_parameters(row) for _, row in prior_samples_df.iterrows()]

priors_payload = {
    "batch_name": batch_name,
    "parameter_grid": BASELINE_PARAMETER_GRID,
    "baseline_parameter_grid": BASELINE_PARAMETER_GRID,
    "baseline_parameter_uniform_specs": BASELINE_PARAMETER_UNIFORM_SPECS,
    "normalized_baseline_parameter_uniform_specs": normalized_baseline_parameter_uniform_specs,
    "n_uniform_samples_per_grid_row": N_UNIFORM_SAMPLES_PER_GRID_ROW,
    "parameter_set_sampling_seed": PARAMETER_SET_SAMPLING_SEED,
    "step_current_amplitudes_nA": step_current_amplitudes_nA,
    "include_zero_step_reference": INCLUDE_ZERO_STEP_REFERENCE,
    "fixed_parameters": FIXED_PARAMETERS,
    "baseline_parameter_sets_df": baseline_parameter_sets_df,
    "parameter_sets_df": parameter_sets_df,
    "paired_repeat_seed_df": paired_repeat_seed_df,
    "prior_samples_df": prior_samples_df,
    "repeats_per_parameter_set": repeats_per_parameter_set,
    "batch_repeat_seed": batch_repeat_seed,
}
priors_path = Path(batch_name) / f"{Path(batch_name).name}_priors.pkl"
with open(priors_path, "wb") as f:
    pickle.dump(priors_payload, f)
print(f"Saved priors to {priors_path}")
print(f"Built {len(params_prior_list)} SimulationParameters objects")


In [ ]:
# # Check for existing corrupted simulation outputs # #
# import h5py
# import pandas as pd
# from pathlib import Path

# run_output_df = pd.read_pickle(Path(batch_name) / "step_response_run_outputs.pkl")

# bad = []
# for path in run_output_df["simulation_output_h5"]:
#     try:
#         with h5py.File(path, "r") as f:
#             _ = f["step_response_diagnostics"]
#     except Exception as exc:
#         bad.append({"path": path, "error": str(exc)})

# bad_df = pd.DataFrame(bad)
# display(bad_df)


In [ ]:
# Optional execution cell.
# Use joblib/loky here rather than concurrent.futures.ProcessPoolExecutor.
# This matches the main batch notebook and is more robust for notebook-defined
# worker functions on Windows. Each worker resets Brian2 after a run so state
# from one simulation does not leak into the next simulation in that process.
def _run_and_reset(params: SimulationParameters):
    try:
        return run_simulation(params)
    finally:
        device.reinit()
        device.activate()


def parallel_simulate(params_prior_list, n_jobs: int = 1):
    n_jobs = int(n_jobs)
    if n_jobs <= 1:
        outputs = []
        for i, params in enumerate(params_prior_list, start=1):
            print(f"Running simulation {i}/{len(params_prior_list)}")
            outputs.append(_run_and_reset(params))
        return outputs
    return Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
        delayed(_run_and_reset)(params) for params in params_prior_list
    )


run_output_df = None
if RUN_SIMULATIONS:
    outputs = parallel_simulate(params_prior_list, n_jobs=SIM_PARALLEL_CPUS)
    run_output_df = prior_samples_df.copy()
    run_output_df["simulation_output_h5"] = [str(path) for path in outputs]
    run_output_csv = Path(batch_name) / "step_response_run_outputs.csv"
    run_output_pkl = Path(batch_name) / "step_response_run_outputs.pkl"
    run_output_df.to_csv(run_output_csv, index=False)
    run_output_df.to_pickle(run_output_pkl)
    print(f"All requested simulations complete. Saved run output table to {run_output_pkl}")
elif LOAD_EXISTING_RUN_OUTPUT_TABLE_WHEN_NOT_SIMULATING:
    run_output_pkl = Path(batch_name) / "step_response_run_outputs.pkl"
    if run_output_pkl.exists():
        run_output_df = pd.read_pickle(run_output_pkl)
        print(f"RUN_SIMULATIONS is False; loaded existing run output table from {run_output_pkl}")
    else:
        print(f"RUN_SIMULATIONS is False and no run output table was found at {run_output_pkl}")
else:
    print("RUN_SIMULATIONS is False; parameter tables and SimulationParameters objects are ready.")

if run_output_df is not None and FINALIZE_STEP_RESPONSE_REPEATS:
    finalization_summary_df = finalize_step_response_batch(
        run_output_df,
        n_jobs=FINALIZATION_N_JOBS,
        regenerate_figures=REGENERATE_STEP_RESPONSE_FIGURES_ON_FINALIZE,
    )
    finalization_csv = Path(batch_name) / "step_response_finalization_summary.csv"
    finalization_pkl = Path(batch_name) / "step_response_finalization_summary.pkl"
    finalization_summary_df.to_csv(finalization_csv, index=False)
    finalization_summary_df.to_pickle(finalization_pkl)
    print(f"Finalized matched step-response repeats. Saved summary to {finalization_pkl}")
    display(finalization_summary_df.head())

if run_output_df is not None and FINALIZE_STEP_RESPONSE_MEAN_TRACES:
    mean_trace_summary_df = finalize_step_response_batch_mean_traces(
        run_output_df,
        n_jobs=MEAN_TRACE_FINALIZATION_N_JOBS,
        regenerate_figures=REGENERATE_MEAN_TRACE_FIGURES,
        plot_style=MEAN_TRACE_DIAGNOSTIC_PLOT_STYLE,
    )
    mean_trace_csv = Path(batch_name) / "step_response_mean_trace_finalization_summary.csv"
    mean_trace_pkl = Path(batch_name) / "step_response_mean_trace_finalization_summary.pkl"
    mean_trace_summary_df.to_csv(mean_trace_csv, index=False)
    mean_trace_summary_df.to_pickle(mean_trace_pkl)
    print(f"Finalized parameter-set mean traces. Saved summary to {mean_trace_pkl}")
    display(mean_trace_summary_df.head())
